In [0]:
from pyspark.sql import functions as F

# Simulate a customer change
updated_customer = spark.sql("""
SELECT
    customer_id,
    first_name,
    last_name,
    email,
    city,
    state,
    country,
    signup_date,
    'Regular' AS customer_segment,
    current_timestamp() AS updated_at
FROM workspace.silver.customers_scd2
WHERE customer_id = 'C001'
  AND is_current = true
""")

display(updated_customer)

In [0]:
%sql
UPDATE workspace.silver.customers_scd2
SET is_current = false,
    effective_end_date = current_timestamp()
WHERE customer_id = 'C001'
  AND is_current = true;

In [0]:
%sql
INSERT INTO workspace.silver.customers_scd2
SELECT
    customer_id,
    first_name,
    last_name,
    email,
    city,
    state,
    country,
    signup_date,
    'Regular' AS customer_segment,
    current_timestamp() AS updated_at,
    current_timestamp() AS _ingestion_timestamp,
    'manual_insert' AS _source_file,
    current_timestamp() AS effective_start_date,
    TIMESTAMP '9999-12-31 23:59:59' AS effective_end_date,
    true AS is_current
FROM workspace.silver.customers_scd2
WHERE customer_id = 'C001'
ORDER BY effective_start_date DESC
LIMIT 1;

In [0]:
%sql
SELECT
    customer_id,
    customer_segment,
    effective_start_date,
    effective_end_date,
    is_current
FROM workspace.silver.customers_scd2
WHERE customer_id = 'C001'
ORDER BY effective_start_date;